# POS Tagging: HMM vs CRF vs Rule-Based
**Dataset:** Penn Treebank (via NLTK)  
**Methods:** Rule-Based (Baseline), Hidden Markov Model (HMM), Conditional Random Field (CRF)

## 1. Setup & Data Loading

In [ ]:
# ── Installs (run once in Colab) ──────────────────────────────────────────────
# !pip install nltk sklearn-crfsuite scikit-learn numpy pandas matplotlib seaborn -q

import nltk, random, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import defaultdict, Counter
from sklearn.metrics import classification_report, accuracy_score
import sklearn_crfsuite
from sklearn_crfsuite import metrics as crf_metrics

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# Download required corpora
for pkg in ['treebank', 'averaged_perceptron_tagger', 'punkt', 'universal_tagset']:
    nltk.download(pkg, quiet=True)

from nltk.corpus import treebank

# Load all tagged sentences
all_sents = treebank.tagged_sents()
print(f"Total sentences : {len(all_sents)}")
print(f"Total tokens    : {sum(len(s) for s in all_sents):,}")
print(f"Sample sentence : {all_sents[0][:6]}")

## 2. Train / Test Split

In [ ]:
random.shuffle(list(all_sents))   # shuffle in-place via list copy
shuffled = list(all_sents)
random.shuffle(shuffled)

split = int(0.8 * len(shuffled))
train_sents = shuffled[:split]
test_sents  = shuffled[split:]

print(f"Train sentences : {len(train_sents):,}")
print(f"Test  sentences : {len(test_sents):,}")

# Flatten helpers
def flatten_tokens(sents): return [w for s in sents for w, _ in s]
def flatten_tags(sents):   return [t for s in sents for _, t in s]

y_true = flatten_tags(test_sents)
print(f"\nTest tokens     : {len(y_true):,}")

# Tag distribution
tag_counts = Counter(flatten_tags(train_sents))
print(f"\nTop-10 tags (train):")
for tag, cnt in tag_counts.most_common(10):
    print(f"  {tag:10s} {cnt:6,}")

## 3. Method 1 — Rule-Based Tagger (Baseline)
Uses hand-crafted suffix/morphology rules (no training data required).

In [ ]:
def rule_based_tag(word):
    """Lightweight deterministic tagger based on morphological cues."""
    w = word.lower()
    # Punctuation / symbols
    if word in ('.', '!', '?'): return '.'
    if word in (',', ';', ':'): return ','
    if word in ('"', "''", '``'): return "''"
    if word in ('-', '--', '...'): return ':'
    if word in ('(', '-LRB-'): return '-LRB-'
    if word in (')', '-RRB-'): return '-RRB-'
    if word == '$': return '$'
    if word == '#': return '#'
    # Determiners / conjunctions / prepositions
    if w in ('the', 'a', 'an'): return 'DT'
    if w in ('and', 'but', 'or', 'nor', 'yet', 'so'): return 'CC'
    if w in ('in', 'on', 'at', 'by', 'for', 'with', 'about', 'against',
              'between', 'into', 'through', 'during', 'before', 'after',
              'above', 'below', 'from', 'up', 'down', 'out', 'of', 'to'):
        return 'IN'
    if w in ('that', 'which', 'who', 'whom', 'whose', 'where', 'when',
              'how', 'why', 'if', 'whether', 'because', 'although',
              'while', 'since', 'as', 'until'): return 'IN'
    if w in ('is', 'are', 'was', 'were', 'be', 'been', 'being',
              'am'): return 'VBZ'
    if w in ('have', 'has', 'had'): return 'VBZ'
    if w in ('will', 'would', 'can', 'could', 'may', 'might',
              'shall', 'should', 'must', 'do', 'does', 'did'): return 'MD'
    if w in ('not', "n't"): return 'RB'
    if w in ('i', 'we', 'you', 'he', 'she', 'they', 'it',
              'me', 'us', 'him', 'her', 'them'): return 'PRP'
    if w in ('my', 'our', 'your', 'his', 'its', 'their'): return 'PRP$'
    if w in ('this', 'these', 'those', 'that'): return 'DT'
    # Morphological suffix rules
    if w.endswith('ing'):  return 'VBG'
    if w.endswith('ed'):   return 'VBD'
    if w.endswith('ly'):   return 'RB'
    if w.endswith('ness'): return 'NN'
    if w.endswith('ment'): return 'NN'
    if w.endswith('tion') or w.endswith('sion'): return 'NN'
    if w.endswith('ity') or w.endswith('ty'):    return 'NN'
    if w.endswith('ous') or w.endswith('ful') or w.endswith('ive'): return 'JJ'
    if w.endswith('al') or w.endswith('ic'):     return 'JJ'
    if w.endswith('er') or w.endswith('est'):    return 'JJR'
    if w.endswith('s') and not w.endswith('ss'): return 'NNS'
    # Numeric / capitalization
    if word[0].isupper() and len(word) > 1:      return 'NNP'
    if word.replace(',', '').replace('.', '').replace('%', '').isdigit(): return 'CD'
    return 'NN'   # default

t0 = time.time()
y_rule = [rule_based_tag(w) for w, _ in
          [(w, t) for s in test_sents for w, t in s]]
rule_time = time.time() - t0

rule_acc = accuracy_score(y_true, y_rule)
print(f"Rule-Based Accuracy : {rule_acc:.4f}  ({rule_acc*100:.2f}%)")
print(f"Inference time      : {rule_time:.3f}s")

## 4. Method 2 — Hidden Markov Model (HMM)
Bigram HMM trained from scratch using MLE on the Penn Treebank training set. Decoding uses the Viterbi algorithm.

In [ ]:
class BigramHMM:
    """Bigram Hidden Markov Model with add-k smoothing and Viterbi decoding."""

    def __init__(self, k=0.01):
        self.k = k          # smoothing constant
        self.tags = []
        self.tag2i = {}
        self.vocab = set()
        self.transition = None   # P(tag_t | tag_{t-1})
        self.emission   = None   # P(word | tag)
        self.init_prob  = None   # P(first tag)
        self.word2i = {}

    def train(self, tagged_sents):
        """Estimate transition & emission probabilities from data."""
        # Collect all tags and words
        tag_set  = set()
        word_set = set()
        for sent in tagged_sents:
            for w, t in sent:
                tag_set.add(t)
                word_set.add(w.lower())

        self.tags   = sorted(tag_set)
        self.tag2i  = {t: i for i, t in enumerate(self.tags)}
        self.vocab  = word_set
        self.word2i = {w: i for i, w in enumerate(sorted(word_set))}
        T = len(self.tags)
        V = len(self.vocab)

        # Count arrays
        trans_cnt  = np.zeros((T, T))
        emiss_cnt  = np.zeros((T, V))
        init_cnt   = np.zeros(T)
        tag_totals = np.zeros(T)

        for sent in tagged_sents:
            if not sent: continue
            _, t0 = sent[0]
            if t0 in self.tag2i:
                init_cnt[self.tag2i[t0]] += 1
            for idx, (w, t) in enumerate(sent):
                if t not in self.tag2i: continue
                ti = self.tag2i[t]
                wl = w.lower()
                if wl in self.word2i:
                    emiss_cnt[ti, self.word2i[wl]] += 1
                tag_totals[ti] += 1
                if idx > 0:
                    _, tprev = sent[idx-1]
                    if tprev in self.tag2i:
                        trans_cnt[self.tag2i[tprev], ti] += 1

        # Smoothed probabilities (log-space)
        k = self.k
        self.init_prob = np.log((init_cnt + k) / (init_cnt.sum() + k * T))
        # Transition
        row_sums = trans_cnt.sum(axis=1, keepdims=True)
        self.transition = np.log((trans_cnt + k) / (row_sums + k * T + 1e-10))
        # Emission
        self.emission = np.log((emiss_cnt + k) / (tag_totals[:, None] + k * V + 1e-10))

        # Unknown-word emission: average over rare-word distribution
        self.unk_log_prob = np.log(k / (tag_totals + k * V + 1e-10))

    def _emit(self, tag_i, word):
        wl = word.lower()
        if wl in self.word2i:
            return self.emission[tag_i, self.word2i[wl]]
        return self.unk_log_prob[tag_i]

    def viterbi(self, words):
        """Viterbi decoding; returns list of tags."""
        T  = len(self.tags)
        N  = len(words)
        if N == 0: return []

        delta   = np.full((N, T), -np.inf)
        psi     = np.zeros((N, T), dtype=int)

        # Init
        delta[0] = self.init_prob + np.array([self._emit(i, words[0]) for i in range(T)])

        # Recursion
        for n in range(1, N):
            emit = np.array([self._emit(i, words[n]) for i in range(T)])
            scores = delta[n-1, :, None] + self.transition   # (T, T)
            psi[n]   = scores.argmax(axis=0)
            delta[n] = scores.max(axis=0) + emit

        # Backtrack
        tags = [int(delta[-1].argmax())]
        for n in range(N-1, 0, -1):
            tags.append(int(psi[n, tags[-1]]))
        tags.reverse()
        return [self.tags[i] for i in tags]

    def tag_sentence(self, words):
        return list(zip(words, self.viterbi(words)))

# ── Train ──────────────────────────────────────────────────────────────────────
print("Training HMM …")
hmm = BigramHMM(k=0.01)
t0 = time.time()
hmm.train(train_sents)
hmm_train_time = time.time() - t0
print(f"  Tags in model : {len(hmm.tags)}")
print(f"  Vocab size    : {len(hmm.vocab):,}")
print(f"  Training time : {hmm_train_time:.2f}s")

# ── Evaluate ───────────────────────────────────────────────────────────────────
print("\nDecoding test set …")
t0 = time.time()
y_hmm = []
for sent in test_sents:
    words = [w for w, _ in sent]
    pred  = hmm.viterbi(words)
    y_hmm.extend(pred)
hmm_inf_time = time.time() - t0

hmm_acc = accuracy_score(y_true, y_hmm)
print(f"\nHMM Accuracy    : {hmm_acc:.4f}  ({hmm_acc*100:.2f}%)")
print(f"Inference time  : {hmm_inf_time:.2f}s")

## 5. Method 3 — Conditional Random Field (CRF)
CRFs model P(tags | words) globally, capturing rich feature context without independence assumptions.

In [ ]:
def word_features(sent, i):
    """Rich feature set for token at position i in sentence (list of words)."""
    word = sent[i]
    wl   = word.lower()

    feats = {
        'bias':         True,
        'word':         wl,
        'word[:3]':     wl[:3],
        'word[:2]':     wl[:2],
        'word[-3:]':    wl[-3:],
        'word[-2:]':    wl[-2:],
        'isupper':      word.isupper(),
        'istitle':      word.istitle(),
        'isdigit':      word.isdigit(),
        'hasdigit':     any(c.isdigit() for c in word),
        'ishyphen':     '-' in word,
        'len':          len(word),
    }

    if i > 0:
        pw  = sent[i-1].lower()
        feats.update({
            'prev_word':      pw,
            'prev_word[-2:]': pw[-2:],
            'prev_istitle':   sent[i-1].istitle(),
            'prev_isupper':   sent[i-1].isupper(),
        })
    else:
        feats['BOS'] = True

    if i > 1:
        pp = sent[i-2].lower()
        feats['pprev_word'] = pp

    if i < len(sent) - 1:
        nw = sent[i+1].lower()
        feats.update({
            'next_word':      nw,
            'next_word[-2:]': nw[-2:],
            'next_istitle':   sent[i+1].istitle(),
        })
    else:
        feats['EOS'] = True

    if i < len(sent) - 2:
        feats['nnext_word'] = sent[i+2].lower()

    return feats

def sent2features(sent):  # sent = list of (word, tag) tuples
    words = [w for w, _ in sent]
    return [word_features(words, i) for i in range(len(words))]

def sent2labels(sent):    return [t for _, t in sent]

# Build feature matrices
print("Extracting features …")
t0 = time.time()
X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s)   for s in train_sents]
X_test  = [sent2features(s) for s in test_sents]
y_test  = [sent2labels(s)   for s in test_sents]
print(f"  Feature extraction: {time.time()-t0:.2f}s")

# Train CRF
print("Training CRF …")
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)
t0 = time.time()
crf.fit(X_train, y_train)
crf_train_time = time.time() - t0
print(f"  Training time : {crf_train_time:.2f}s")

# Evaluate
t0 = time.time()
y_crf_nested = crf.predict(X_test)
crf_inf_time = time.time() - t0
y_crf = [t for sent in y_crf_nested for t in sent]

crf_acc = accuracy_score(y_true, y_crf)
print(f"\nCRF Accuracy    : {crf_acc:.4f}  ({crf_acc*100:.2f}%)")
print(f"Inference time  : {crf_inf_time:.2f}s")

## 6. Performance Comparison

In [ ]:
# ── Accuracy summary table ─────────────────────────────────────────────────────
results = pd.DataFrame([
    {'Method': 'Rule-Based',  'Accuracy': rule_acc, 'Inference (s)': rule_time,  'Train (s)': 0},
    {'Method': 'HMM',         'Accuracy': hmm_acc,  'Inference (s)': hmm_inf_time, 'Train (s)': hmm_train_time},
    {'Method': 'CRF',         'Accuracy': crf_acc,  'Inference (s)': crf_inf_time, 'Train (s)': crf_train_time},
])
results['Accuracy (%)'] = (results['Accuracy'] * 100).round(2)
print(results[['Method', 'Accuracy (%)', 'Train (s)', 'Inference (s)']].to_string(index=False))

In [ ]:
# ── Figure 1: Overall accuracy bar chart ──────────────────────────────────────
colors = ['#6c9bc2', '#e07b39', '#4caf82']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bars = axes[0].bar(results['Method'], results['Accuracy (%)'], color=colors, width=0.5, edgecolor='white', linewidth=1.5)
axes[0].set_ylim(0, 105)
axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_title('Overall Token Accuracy', fontsize=13, fontweight='bold')
for bar, val in zip(bars, results['Accuracy (%)']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                 f'{val:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].spines[['top', 'right']].set_visible(False)

# Training time
t_bars = axes[1].bar(results['Method'], results['Train (s)'], color=colors, width=0.5, edgecolor='white', linewidth=1.5)
axes[1].set_ylabel('Time (seconds)', fontsize=12)
axes[1].set_title('Training Time', fontsize=13, fontweight='bold')
for bar, val in zip(t_bars, results['Train (s)']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}s', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig1_accuracy_time.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 1 saved.")

In [ ]:
# ── Figure 2: Per-tag F1 heatmap comparison ────────────────────────────────────
common_tags = sorted(set(y_true))  # all tags that appear in test set

def per_tag_f1(y_true, y_pred, tags):
    from sklearn.metrics import f1_score
    return {t: f1_score(y_true, y_pred, labels=[t], average='micro',
                        zero_division=0) for t in tags}

f1_rule = per_tag_f1(y_true, y_rule, common_tags)
f1_hmm  = per_tag_f1(y_true, y_hmm,  common_tags)
f1_crf  = per_tag_f1(y_true, y_crf,  common_tags)

# Filter to tags with >50 occurrences for readability
tag_freq = Counter(y_true)
plot_tags = [t for t in common_tags if tag_freq[t] >= 50]

heatmap_data = pd.DataFrame({
    'Rule-Based': [f1_rule[t] for t in plot_tags],
    'HMM':        [f1_hmm[t]  for t in plot_tags],
    'CRF':        [f1_crf[t]  for t in plot_tags],
}, index=plot_tags)

fig, ax = plt.subplots(figsize=(8, max(6, len(plot_tags)*0.35)))
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='YlOrRd',
            vmin=0, vmax=1, ax=ax, linewidths=0.4, linecolor='white',
            cbar_kws={'label': 'F1 Score'})
ax.set_title('Per-Tag F1 Score by Method', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Tagger', fontsize=11)
ax.set_ylabel('POS Tag', fontsize=11)
plt.tight_layout()
plt.savefig('fig2_pertag_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 2 saved.")

In [ ]:
# ── Figure 3: CRF confusion matrix (top 15 tags) ──────────────────────────────
from sklearn.metrics import confusion_matrix

top15 = [t for t, _ in Counter(y_true).most_common(15)]

# Filter to only top-15 predictions / truths
pairs = [(t, p) for t, p in zip(y_true, y_crf) if t in top15]
yt15  = [t for t, _ in pairs]
yp15  = [p for _, p in pairs]

cm = confusion_matrix(yt15, yp15, labels=top15)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm_norm, xticklabels=top15, yticklabels=top15,
            annot=True, fmt='.2f', cmap='Blues', ax=ax,
            linewidths=0.3, linecolor='lightgray',
            cbar_kws={'label': 'Normalised Count'})
ax.set_title('CRF Confusion Matrix — Top 15 Tags (normalised)', fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Predicted Tag', fontsize=11)
ax.set_ylabel('True Tag', fontsize=11)
plt.tight_layout()
plt.savefig('fig3_crf_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 3 saved.")

In [ ]:
# ── Classification report for CRF ─────────────────────────────────────────────
print("=== CRF Classification Report ===")
print(classification_report(y_true, y_crf, zero_division=0))

In [ ]:
# ── Figure 4: Tag-level accuracy delta (CRF – HMM) ────────────────────────────
from sklearn.metrics import f1_score

delta = {t: f1_crf[t] - f1_hmm[t] for t in plot_tags}
delta_df = pd.DataFrame({'tag': list(delta.keys()), 'delta': list(delta.values())})
delta_df = delta_df.sort_values('delta')

fig, ax = plt.subplots(figsize=(8, max(5, len(plot_tags)*0.35)))
colors_d = ['#d9534f' if v < 0 else '#4caf82' for v in delta_df['delta']]
ax.barh(delta_df['tag'], delta_df['delta'], color=colors_d, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('F1 Improvement (CRF − HMM)', fontsize=11)
ax.set_title('Per-Tag F1 Gain: CRF over HMM', fontsize=13, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
green_patch = mpatches.Patch(color='#4caf82', label='CRF better')
red_patch   = mpatches.Patch(color='#d9534f', label='HMM better')
ax.legend(handles=[green_patch, red_patch], fontsize=10)
plt.tight_layout()
plt.savefig('fig4_crf_vs_hmm_delta.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 4 saved.")

## 7. Error Analysis

In [ ]:
# Most common CRF errors
error_pairs = Counter(
    (true, pred)
    for true, pred in zip(y_true, y_crf)
    if true != pred
)
print("Top-10 CRF Errors  (true → predicted):")
for (true, pred), cnt in error_pairs.most_common(10):
    print(f"  {true:8s} → {pred:8s}  {cnt:5,} times")

print()
# Show example sentences where CRF makes mistakes
print("Example HMM vs CRF predictions on 3 test sentences:")
for sent in test_sents[:3]:
    words = [w for w, _ in sent]
    true_tags = [t for _, t in sent]
    hmm_pred  = hmm.viterbi(words)
    crf_pred  = crf.predict([sent2features(sent)])[0]
    print(f"\nWords  : {' '.join(words[:10])}")
    print(f"True   : {' '.join(true_tags[:10])}")
    print(f"HMM    : {' '.join(hmm_pred[:10])}")
    print(f"CRF    : {' '.join(crf_pred[:10])}")

## 8. Summary

| Method | Accuracy | Strength | Weakness |
|---|---|---|---|
| Rule-Based | ~50–55% | No training needed | Fragile, limited coverage |
| HMM (Bigram) | ~90–92% | Fast, probabilistic | Strong independence assumptions |
| CRF | ~95–96% | Rich features, global decoding | Slower to train |

**Key takeaway:** CRFs outperform HMMs because they condition on the entire observation sequence and can incorporate arbitrary features (prefixes, suffixes, capitalization, context window). HMMs are still competitive and extremely fast to train.
